# 1. Propósito

Cerrar las cuatro palabras del primer pivote sólo hasta el factor anterior a $W_{1,2}^{+}$. Se preservan orden, fuerza lógica y clases; no se estudia Wiener--Hopf ni Fredholmness.

In [1]:
import sympy as sp
from symbolic_operator_calculus import (
    ClosureObligationStatus, MinimalLemmaChoice,
    build_first_pivot_closure_analysis,
    build_first_pivot_left_core_closure,
    render_first_pivot_left_cores_latex,
    render_left_dilation_covariance_latex,
    render_minimal_closure_decision_yaml,
)
lam, gamma, x = sp.symbols('lambda gamma x')
trace = build_first_pivot_left_core_closure(lam, gamma, x)
cores = {core.identifier: core for core in trace.cores}
print('Phase R loaded:', list(cores))

Phase R loaded: ['L-+', 'L++', 'L-Ghat', 'L+Ghat']


# 2. Convenciones

La transformada usa $x^{-i\lambda}$, la cuantización usa $(x/y)^{i\lambda}$ y $D_\gamma f(x)=f(\gamma x)$.

In [2]:
print('Mellin: x**(-I*lambda); kernel: (x/y)**(I*lambda)')
print('P^-P^+ = P^+P^- != 0; no complementary-projection rule is used')

Mellin: x**(-I*lambda); kernel: (x/y)**(I*lambda)
P^-P^+ = P^+P^- != 0; no complementary-projection rule is used


# 3. Covarianza izquierda

La evaluación en $x/\gamma$ produce $a_\gamma(x,\lambda)=a(x/\gamma,\lambda)$ y la frecuencia inversa $\gamma^{-i\lambda}$.

In [3]:
covariance = trace.covariance
assert covariance.inverse_factor.expression == sp.Pow(gamma, -sp.I*lam, evaluate=False)
assert covariance.frequency_cancellation == 1
print(render_left_dilation_covariance_latex(covariance))

D_{\gamma}^{-1}\operatorname{Op}(a)=\operatorname{Op}_{\mathrm{right}\text{-}\gamma^{-1}}(a_{\gamma},d_{\gamma^{-1}}),\quadD_{\gamma}^{-1}\operatorname{Op}(a)D_{\gamma}=\operatorname{Op}(a_{\gamma}),\quada_{\gamma}(x,\lambda)=a(x/\gamma,\lambda),\quadd_{\gamma^{-1}}(\lambda)=\gamma^{-i\lambda}.


# 4. Invariancia por escalamiento

Se verifican por separado todos los contratos de $\widetilde{\mathcal E}$; la membresía del símbolo radialmente escalado es estándar.

In [4]:
scaling = covariance.scaling
assert scaling.radial_argument == x/gamma
print('status:', scaling.status.value)
print('radial argument:', scaling.radial_argument)
print('contracts:', list(scaling.verified_contracts))

status: ANALYTICALLY_PROVED
radial argument: x/gamma
contracts: ['the C_b(V) norm is unchanged because x -> x/gamma is bijective', 'V-valued continuity is preserved by composition', 'cm_t(a_gamma)=cm_(t/gamma)(a) at zero and infinity', 'covariable-translation suprema are unchanged', 'uniform derivative-tail suprema are unchanged', 'endpoint fibers are preserved by the radial scaling automorphism']


# 5. Semiproducto con $P^\pm$

El Teorema 3.3 se aplica sólo a símbolos estándar en $\mathcal E$ y da fuerza módulo compactos.

In [5]:
print([(rule.name, rule.certification_status.value) for rule in trace.cauchy_rules])
print('source:', trace.cauchy_relations[0].evidence.theorem)
print('checksum:', trace.cauchy_relations[0].evidence.checksum)

[('cauchy_plus_mellin_semiproduct', 'CERTIFIED_MOD_COMPACT'), ('cauchy_minus_mellin_semiproduct', 'CERTIFIED_MOD_COMPACT')]
source: KKL2014TwoShifts Theorem 3.3
checksum: 3f1b91576171681ff3b927685664c169134948dda515dec3737a72fc7a7ae1ec


# 6. $L_{-+}$

$P^-\operatorname{Op}(r_1)D_{\gamma_1}$ queda como Mellin PDO factorado a la derecha módulo compactos.

In [6]:
core = cores['L-+']
print(core.ast_product)
print(core.output_kind, core.output_symbol.oscillatory_factor.expression)
print('residuals:', core.compact_residual_terms)

Product(factors=(OperatorAtom(name='P_minus', kind='projection'), OperatorAtom(name='R1', kind='formal_regularizer'), OperatorAtom(name='U1', kind='dilation_operator')))
RIGHT_FACTORIZED_MELLIN_PDO gamma**(I*lambda)
residuals: ('K_{-,r} D_gamma',)


# 7. $L_{++}$

Las frecuencias se cancelan exactamente bajo la conjugación y queda el símbolo estándar $p^+(\lambda)r_1(x/\gamma_1,\lambda)$.

In [7]:
core = cores['L++']
print(core.ast_product)
print(core.output_kind, core.output_symbol.name)
print('residuals:', core.compact_residual_terms)

Product(factors=(OperatorAtom(name='U1_inverse', kind='inverse_shift_operator'), OperatorAtom(name='P_plus', kind='projection'), OperatorAtom(name='R1', kind='formal_regularizer'), OperatorAtom(name='U1', kind='dilation_operator')))
STANDARD_MELLIN_PDO p^+(lambda) r_1(x/gamma,lambda)
residuals: ('D_gamma^{-1} K_{+,r} D_gamma',)


# 8. $L_{-\widehat G}$

El residuo total suma el residuo coeficiente y el semiproducto izquierdo, ambos compactos tras multiplicación acotada.

In [8]:
core = cores['L-Ghat']
print(core.ast_product)
print(core.output_kind, core.output_symbol.name)
print('residuals:', core.compact_residual_terms)

Product(factors=(OperatorAtom(name='P_minus', kind='projection'), OperatorAtom(name='R1', kind='formal_regularizer'), OperatorAtom(name='Ghat1', kind='normalized_coefficient')))
STANDARD_MELLIN_PDO p^-(lambda) r_1(x,lambda) Ghat_1(x)
residuals: ('P^- K_{r,Ghat}', 'K_{-,rGhat}')


# 9. $L_{+\widehat G}$

La acción exterior introduce $x/\gamma_1$ y $d_{\gamma_1^{-1}}$; el resultado permanece factorado.

In [9]:
core = cores['L+Ghat']
print(core.ast_product)
print(core.output_kind, core.output_symbol.standard_factor.name)
print('frequency:', core.output_symbol.oscillatory_factor.expression)
print('residuals:', core.compact_residual_terms)

Product(factors=(OperatorAtom(name='U1_inverse', kind='inverse_shift_operator'), OperatorAtom(name='P_plus', kind='projection'), OperatorAtom(name='R1', kind='formal_regularizer'), OperatorAtom(name='Ghat1', kind='normalized_coefficient')))
RIGHT_FACTORIZED_MELLIN_PDO p^+(lambda) r_1(x/gamma,lambda) Ghat_1(x/gamma)
frequency: gamma**(-I*lambda)
residuals: ('D_gamma^{-1} P^+ K_{r,Ghat}', 'D_gamma^{-1} K_{+,rGhat}')


# 10. Traducción ponderada

Las cuatro fórmulas se conjugan por $\Phi_\delta^{-1}(\cdot)\Phi_\delta$ sin alterar el orden ni la fuerza módulo compactos.

In [10]:
print(render_first_pivot_left_cores_latex(trace))
assert all(core.weighted_conjugation for core in trace.cores)

\[
\begin{aligned}
P^{-}\,R_{1}\,U_{1} &\simeq \Phi_\delta^{-1}\operatorname{Op}_{\mathrm{right}\text{-}\gamma_1}(p^-r_1,d_{\gamma_1})\Phi_\delta \\
U_{1}^{-1}\,P^{+}\,R_{1}\,U_{1} &\simeq \Phi_\delta^{-1}\operatorname{Op}(p^+(\lambda)r_1(x/\gamma_1,\lambda))\Phi_\delta \\
P^{-}\,R_{1}\,\widehat G_{1} &\simeq \Phi_\delta^{-1}\operatorname{Op}(p^-(\lambda)r_1(x,\lambda)\widehat G_1(x))\Phi_\delta \\
U_{1}^{-1}\,P^{+}\,R_{1}\,\widehat G_{1} &\simeq \Phi_\delta^{-1}\operatorname{Op}_{\mathrm{right}\text{-}\gamma_1^{-1}}(p^+r_{1,\gamma_1}\widehat G_{1,\gamma_1},d_{\gamma_1^{-1}})\Phi_\delta
\end{aligned}
\]



# 11. Obligaciones

P-02/P-03 se conservan; P-04/P-06 se descargan a sus fuerzas explícitas.

In [11]:
closure = build_first_pivot_closure_analysis()
statuses = {item.identifier: item.status.value for item in closure.obligations[:6]}
assert statuses['P-04'] == ClosureObligationStatus.ANALYTICALLY_PROVED.value
assert statuses['P-06'] == ClosureObligationStatus.ANALYTICALLY_PROVED.value
print(statuses)

{'P-01': 'BLOCKED', 'P-02': 'ANALYTICALLY_PROVED', 'P-03': 'ANALYTICALLY_PROVED', 'P-04': 'ANALYTICALLY_PROVED', 'P-05': 'BLOCKED', 'P-06': 'ANALYTICALLY_PROVED'}


# 12. Bloqueo restante

La fase no identifica ni compone el bloque localizado; sólo registra que éste sigue a los cuatro prefijos reducidos.

In [12]:
print('remaining blocker:', trace.remaining_blocker)
print('compact ideal proof:', list(trace.compact_ideal_proof))

remaining blocker: right composition of each reduced core with the still-unidentified Wplus_12
compact ideal proof: ['AKB is compact whenever K is compact and A,B are bounded', 'finite sums of compact residuals remain compact', 'Phi_delta conjugation preserves compactness']


# 13. Impacto sobre H2/H3

Los cuatro prefijos están controlados, pero H2/H3 siguen bloqueados por identificación y composición derecha; la decisión permanece `NONE`.

In [13]:
assert closure.decision.decision is MinimalLemmaChoice.NONE
print(render_minimal_closure_decision_yaml(closure.decision))

decision: NONE
confidence: high
blocking_obligations:
  - "P-01"
  - "P-05"
evidence:
  - "phase-q:exact-right-dilation-definition-proof"
  - "phase-q:certified-Ghat1-semiproduct"
  - "phase-r:four-left-cores-mod-compact"
  - "paper:normalized-wh-blocks-section-six"
  - "KKL2014TwoShifts:3.3"
  - "KKL2014TwoShifts:4.4"
  - "Karlovich2025Cusps:3.3"
rationale: "All four prefixes before Wplus_12 now have certified standard or right-factorized representations modulo compacts. H2 and H3 still require identification of Wplus_12 and a proved right-composition rule."
prerequisite_statement: "Identify Wplus_12 in a compatible operator class, then control its right composition with each of the four already reduced left cores."

